# Azerbaijani Handwriting Recognition Training

## Architecture: CNN → Transformer → CTC

This notebook implements a production-ready HTR system for Azerbaijani handwritten documents:

1. **CNN Encoder**: Extract visual features from handwritten text images
2. **Transformer Encoder**: Capture long-range dependencies and context
3. **CTC Decoder**: Sequence-to-sequence alignment without explicit segmentation

**Configuration:**
- IMG_HEIGHT = 256 (high quality for complex Azerbaijani script)
- BATCH_SIZE = 2 (small dataset)
- Character-level vocabulary with Azerbaijani diacritics (ə ç ğ ı ö ş ü)

In [ ]:
# Cell 2: All imports
import os
import json
import warnings
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass
import random
import numpy as np
import cv2
from PIL import Image
import pillow_heif
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import pandas as pd
from tqdm.auto import tqdm
import Levenshtein

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Register HEIF opener
pillow_heif.register_heif_opener()

# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# Cell 3: Config class
@dataclass
class Config:
    """Configuration for HTR training"""
    
    # Paths
    data_dir: str = "/Users/ismatsamadov/azeri_handwriting_detection/data"
    images_dir: str = "/Users/ismatsamadov/azeri_handwriting_detection/data/images"
    labels_dir: str = "/Users/ismatsamadov/azeri_handwriting_detection/data/labels"
    output_dir: str = "/Users/ismatsamadov/azeri_handwriting_detection/outputs"
    charts_dir: str = "/Users/ismatsamadov/azeri_handwriting_detection/charts"
    
    # Image preprocessing
    IMG_HEIGHT: int = 256  # High quality for complex scripts
    
    # Training hyperparameters
    BATCH_SIZE: int = 2  # Small dataset
    NUM_EPOCHS: int = 30
    LEARNING_RATE: float = 1e-4
    WEIGHT_DECAY: float = 1e-4
    GRADIENT_CLIP: float = 1.0
    
    # CNN encoder parameters
    cnn_channels: List[int] = None  # [64, 128, 256, 256, 512, 512]
    
    # Transformer encoder parameters (from plan.md)
    d_model: int = 512  # Match CNN output channels
    num_encoder_layers: int = 6  # Transformer depth
    num_heads: int = 8  # Multi-head attention
    dim_feedforward: int = 2048  # FFN hidden dimension
    dropout: float = 0.1
    
    # Training schedule
    warmup_steps: int = 500
    patience: int = 10  # Early stopping patience
    
    # Data split
    train_split: float = 0.7
    val_split: float = 0.15
    test_split: float = 0.15
    
    def __post_init__(self):
        if self.cnn_channels is None:
            self.cnn_channels = [64, 128, 256, 256, 512, 512]
        
        # Create directories
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.charts_dir, exist_ok=True)
        os.makedirs(os.path.join(self.charts_dir, "preprocessing"), exist_ok=True)
        os.makedirs(os.path.join(self.charts_dir, "training"), exist_ok=True)
        os.makedirs(os.path.join(self.charts_dir, "predictions"), exist_ok=True)

config = Config()
print("Configuration:")
print(f"  IMG_HEIGHT: {config.IMG_HEIGHT}")
print(f"  BATCH_SIZE: {config.BATCH_SIZE}")
print(f"  Transformer: d_model={config.d_model}, layers={config.num_encoder_layers}, heads={config.num_heads}")
print(f"  Device: {device}")

In [ ]:
# Cell 4: ImagePreprocessor class with visualize_stages() method
class ImagePreprocessor:
    """Preprocess images for HTR with comprehensive stage visualization"""
    
    def __init__(self, target_height: int = 256):
        self.target_height = target_height
    
    def process(self, image: np.ndarray, visualize: bool = False, img_name: str = "sample") -> Tuple[np.ndarray, Dict]:
        """
        Process image through all preprocessing stages.
        
        Args:
            image: Input image (RGB or grayscale)
            visualize: If True, create visualization of all stages
            img_name: Name for saving visualization
        
        Returns:
            Tuple of (processed_image, stages_dict)
        """
        stages = {}
        
        # Stage 0: Original
        stages['0_original'] = image.copy()
        
        # Stage 1: Grayscale conversion
        if len(image.shape) == 3 and image.shape[2] == 3:
            image_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        else:
            image_gray = image.copy()
        stages['1_grayscale'] = image_gray.copy()
        
        # Stage 2: CLAHE (Contrast Limited Adaptive Histogram Equalization)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        image_clahe = clahe.apply(image_gray)
        stages['2_clahe'] = image_clahe.copy()
        
        # Stage 3: Resize (maintain aspect ratio, height normalize)
        h, w = image_clahe.shape
        new_width = int(self.target_height * w / h)
        image_resized = cv2.resize(
            image_clahe, 
            (new_width, self.target_height), 
            interpolation=cv2.INTER_CUBIC
        )
        stages['3_resized'] = image_resized.copy()
        
        # Stage 4: Normalize to [0, 1]
        image_normalized = image_resized.astype(np.float32) / 255.0
        stages['4_normalized'] = image_normalized.copy()
        
        # Visualization
        if visualize:
            self._visualize_stages(stages, img_name, h, w, new_width)
        
        return image_normalized, stages
    
    def _visualize_stages(self, stages: Dict, img_name: str, orig_h: int, orig_w: int, new_w: int):
        """Create comprehensive visualization of all preprocessing stages"""
        
        fig = plt.figure(figsize=(20, 12))
        gs = GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.3)
        
        stage_names = [
            ('0_original', 'Stage 0: Original Image', 'viridis'),
            ('1_grayscale', 'Stage 1: Grayscale Conversion', 'gray'),
            ('2_clahe', 'Stage 2: CLAHE Enhancement', 'gray'),
            ('3_resized', 'Stage 3: Resized (Height Normalized)', 'gray'),
            ('4_normalized', 'Stage 4: Normalized [0, 1]', 'gray'),
        ]
        
        print("\n" + "="*80)
        print(f"PREPROCESSING PIPELINE VISUALIZATION: {img_name}")
        print("="*80)
        
        for idx, (stage_key, title, cmap) in enumerate(stage_names):
            row = idx // 3
            col = idx % 3
            ax = fig.add_subplot(gs[row, col])
            
            img_data = stages[stage_key]
            
            # Display image
            if stage_key == '0_original' and len(img_data.shape) == 3:
                ax.imshow(img_data)
            else:
                if cmap == 'gray':
                    ax.imshow(img_data, cmap='gray', vmin=0, vmax=255 if img_data.max() > 1 else 1)
                else:
                    ax.imshow(img_data, cmap=cmap)
            
            # Calculate statistics
            shape_str = f"{img_data.shape[0]}×{img_data.shape[1]}" if len(img_data.shape) == 2 else f"{img_data.shape[0]}×{img_data.shape[1]}×{img_data.shape[2]}"
            value_min = img_data.min()
            value_max = img_data.max()
            value_mean = img_data.mean()
            
            # Set title with stats
            ax.set_title(
                f"{title}\nShape: {shape_str}\nRange: [{value_min:.3f}, {value_max:.3f}] | Mean: {value_mean:.3f}",
                fontsize=10,
                pad=10
            )
            ax.axis('off')
            
            # Print to console
            print(f"\n{title}")
            print(f"  Shape: {shape_str}")
            print(f"  Value range: [{value_min:.6f}, {value_max:.6f}]")
            print(f"  Mean: {value_mean:.6f}")
            print(f"  Dtype: {img_data.dtype}")
        
        # Add transformation info in the 6th subplot
        ax_info = fig.add_subplot(gs[1, 2])
        ax_info.axis('off')
        
        info_text = f"""Preprocessing Pipeline Summary

Original Size: {orig_h}×{orig_w}
Target Height: {self.target_height}
Final Size: {self.target_height}×{new_w}

Transformations:
1. RGB → Grayscale (if needed)
2. CLAHE contrast enhancement
   - Clip limit: 2.0
   - Tile size: 8×8
3. Resize with aspect ratio preserved
   - Method: Cubic interpolation
4. Normalize to [0, 1] range

Aspect Ratio Change:
  Original: {orig_w/orig_h:.3f}
  Final: {new_w/self.target_height:.3f}
  Preserved: {abs((new_w/self.target_height) - (orig_w/orig_h)) < 0.01}
"""
        ax_info.text(0.1, 0.5, info_text, fontsize=10, verticalalignment='center',
                    family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.suptitle(f'Preprocessing Stage Visualization: {img_name}', fontsize=16, fontweight='bold', y=0.98)
        
        # Save figure
        save_path = os.path.join(config.charts_dir, "preprocessing", f"{img_name}_stages.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"\n{'='*80}")
        print(f"Visualization saved to: {save_path}")
        print(f"{'='*80}\n")
        plt.show()

# Initialize preprocessor
preprocessor = ImagePreprocessor(target_height=config.IMG_HEIGHT)
print(f"ImagePreprocessor initialized with target_height={config.IMG_HEIGHT}")

In [ ]:
# Cell 5: TEST CELL - Load first image and visualize all preprocessing stages
print("Testing preprocessing pipeline with first sample image...\n")

# Get first HEIC image
image_files = sorted([f for f in os.listdir(config.images_dir) if f.endswith('.HEIC')])
if not image_files:
    raise FileNotFoundError(f"No HEIC images found in {config.images_dir}")

first_img_name = image_files[0]
first_img_path = os.path.join(config.images_dir, first_img_name)

print(f"Loading image: {first_img_path}")

# Load HEIC image
img_pil = Image.open(first_img_path)
img_np = np.array(img_pil)

print(f"Loaded image shape: {img_np.shape}")
print(f"Image dtype: {img_np.dtype}")
print(f"Image range: [{img_np.min()}, {img_np.max()}]")

# Process with visualization
test_name = first_img_name.replace('.HEIC', '')
processed_img, stages = preprocessor.process(img_np, visualize=True, img_name=test_name)

print(f"\nFinal processed image shape: {processed_img.shape}")
print(f"Final processed image dtype: {processed_img.dtype}")
print(f"Final processed image range: [{processed_img.min():.6f}, {processed_img.max():.6f}]")
print("\nPreprocessing test completed successfully!")

In [ ]:
# Cell 6: HTRDataset class (reads labels directly, no arrow delimiter)
class HTRDataset(Dataset):
    """Dataset for Handwritten Text Recognition"""
    
    def __init__(self, image_paths: List[str], label_paths: List[str], 
                 char_to_idx: Dict[str, int], preprocessor: ImagePreprocessor):
        self.image_paths = image_paths
        self.label_paths = label_paths
        self.char_to_idx = char_to_idx
        self.preprocessor = preprocessor
        
        assert len(image_paths) == len(label_paths), "Image and label counts must match"
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load image
        img_pil = Image.open(self.image_paths[idx])
        img_np = np.array(img_pil)
        
        # Preprocess (no visualization during training)
        img_processed, _ = self.preprocessor.process(img_np, visualize=False)
        
        # Convert to tensor: [1, H, W] for grayscale
        img_tensor = torch.from_numpy(img_processed).unsqueeze(0).float()
        
        # Load label - read entire file, labels are plain text (no arrow delimiters)
        with open(self.label_paths[idx], 'r', encoding='utf-8') as f:
            # Read all lines and join them
            label_text = ' '.join(line.strip() for line in f.readlines() if line.strip())
        
        # Encode label to indices
        label_encoded = [self.char_to_idx.get(ch, self.char_to_idx['<UNK>']) for ch in label_text]
        label_tensor = torch.tensor(label_encoded, dtype=torch.long)
        
        return img_tensor, label_tensor, label_text

print("HTRDataset class defined")

In [ ]:
# Cell 7: Build vocabulary from all label files
print("Building vocabulary from label files...\n")

# Collect all characters from labels
all_chars = set()
label_files = sorted([f for f in os.listdir(config.labels_dir) if f.endswith('.txt')])

print(f"Found {len(label_files)} label files")

for label_file in label_files:
    label_path = os.path.join(config.labels_dir, label_file)
    with open(label_path, 'r', encoding='utf-8') as f:
        text = f.read()
        all_chars.update(text)

# Remove line numbers and arrows (artifact from label format)
all_chars = {ch for ch in all_chars if ch not in ['→']}

# Sort for consistent ordering
sorted_chars = sorted(all_chars)

# Create vocabulary with special tokens
# CTC blank is implicit (index 0 in CTCLoss)
special_tokens = ['<PAD>', '<UNK>']  # Padding and unknown
vocab = special_tokens + sorted_chars

char_to_idx = {ch: idx for idx, ch in enumerate(vocab)}
idx_to_char = {idx: ch for ch, idx in char_to_idx.items()}

# CTC needs blank at the end for CTCLoss
num_classes = len(vocab)  # CTCLoss blank will be index num_classes

print(f"\nVocabulary size: {len(vocab)} characters")
print(f"Number of classes (including CTC blank): {num_classes + 1}")
print(f"\nSpecial tokens: {special_tokens}")
print(f"\nAzerbaijani-specific characters found: {[ch for ch in sorted_chars if ch in 'əçğıöşüƏÇĞIÖŞÜ']}")
print(f"\nFirst 50 characters in vocab: {vocab[:50]}")

# Save vocabulary
vocab_path = os.path.join(config.output_dir, 'vocabulary.json')
with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump({
        'char_to_idx': char_to_idx,
        'idx_to_char': idx_to_char,
        'num_classes': num_classes + 1,  # Including CTC blank
        'vocab_size': len(vocab)
    }, f, ensure_ascii=False, indent=2)

print(f"\nVocabulary saved to: {vocab_path}")

In [ ]:
# Cell 8: Create data loaders with collate_fn
def collate_fn(batch):
    """
    Collate function for DataLoader.
    Handles variable-width images and variable-length labels.
    """
    images, labels, texts = zip(*batch)
    
    # Pad images to max width in batch
    max_width = max(img.shape[2] for img in images)
    padded_images = []
    for img in images:
        # img shape: [1, H, W]
        pad_width = max_width - img.shape[2]
        if pad_width > 0:
            img = F.pad(img, (0, pad_width), value=0)  # Pad right
        padded_images.append(img)
    
    images_batch = torch.stack(padded_images, dim=0)  # [B, 1, H, W]
    
    # Pad labels
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=0)  # [B, max_label_len]
    
    # Get lengths
    label_lengths = torch.tensor([len(label) for label in labels], dtype=torch.long)
    
    return images_batch, labels_padded, label_lengths, texts

# Prepare dataset splits
print("Preparing dataset splits...\n")

# Get all image and label paths
image_files = sorted([f for f in os.listdir(config.images_dir) if f.endswith('.HEIC')])
label_files = sorted([f for f in os.listdir(config.labels_dir) if f.endswith('.txt')])

# Match images to labels by numerical prefix
image_label_pairs = []
for img_file in image_files:
    img_num = img_file.split('.')[0]  # e.g., '01'
    # Find corresponding label
    matching_labels = [lf for lf in label_files if img_num in lf]
    if matching_labels:
        img_path = os.path.join(config.images_dir, img_file)
        label_path = os.path.join(config.labels_dir, matching_labels[0])
        image_label_pairs.append((img_path, label_path))

print(f"Found {len(image_label_pairs)} image-label pairs")

# Shuffle and split
random.shuffle(image_label_pairs)

n_total = len(image_label_pairs)
n_train = int(n_total * config.train_split)
n_val = int(n_total * config.val_split)

train_pairs = image_label_pairs[:n_train]
val_pairs = image_label_pairs[n_train:n_train + n_val]
test_pairs = image_label_pairs[n_train + n_val:]

print(f"\nDataset splits:")
print(f"  Train: {len(train_pairs)} samples")
print(f"  Val: {len(val_pairs)} samples")
print(f"  Test: {len(test_pairs)} samples")

# Create datasets
train_images, train_labels = zip(*train_pairs) if train_pairs else ([], [])
val_images, val_labels = zip(*val_pairs) if val_pairs else ([], [])
test_images, test_labels = zip(*test_pairs) if test_pairs else ([], [])

train_dataset = HTRDataset(list(train_images), list(train_labels), char_to_idx, preprocessor)
val_dataset = HTRDataset(list(val_images), list(val_labels), char_to_idx, preprocessor)
test_dataset = HTRDataset(list(test_images), list(test_labels), char_to_idx, preprocessor)

# Create data loaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=config.BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn,
    num_workers=0,  # Set to 0 for HEIC compatibility
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=config.BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=config.BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print(f"\nData loaders created with batch_size={config.BATCH_SIZE}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Cell 9: HTRModel class (CNN + Transformer + CTC from plan.md)
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for Transformer"""
    
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x: [B, T, C]
        return x + self.pe[:, :x.size(1), :]


class HTRModel(nn.Module):
    """
    HTR Model: CNN Encoder → Transformer Encoder → CTC Decoder
    Based on plan.md architecture specification
    """
    
    def __init__(self, num_classes: int, config: Config):
        super().__init__()
        self.config = config
        self.num_classes = num_classes
        
        # CNN Encoder - extract visual features
        # From plan.md: 6 conv blocks reducing height to 1
        self.cnn = nn.Sequential(
            # Block 1: 1 -> 64
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            
            # Block 2: 64 -> 128, reduce height by 2
            nn.Conv2d(64, 128, kernel_size=3, stride=(2, 1), padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            
            # Block 3: 128 -> 256
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            
            # Block 4: 256 -> 256, reduce height by 2
            nn.Conv2d(256, 256, kernel_size=3, stride=(2, 1), padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            
            # Block 5: 256 -> 512
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            
            # Block 6: 512 -> 512, reduce height by 2
            nn.Conv2d(512, 512, kernel_size=3, stride=(2, 1), padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )
        
        # Adaptive pooling to collapse height to 1
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, None))
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(config.d_model)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.num_heads,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=config.num_encoder_layers
        )
        
        # CTC projection head
        self.classifier = nn.Linear(config.d_model, num_classes)
    
    def forward(self, x):
        """
        Args:
            x: [B, 1, H, W] input images
        
        Returns:
            logits: [B, T, num_classes] for CTC loss
        """
        # CNN feature extraction
        features = self.cnn(x)  # [B, 512, H', W']
        
        # Collapse height dimension
        features = self.adaptive_pool(features)  # [B, 512, 1, W']
        features = features.squeeze(2)  # [B, 512, W']
        features = features.permute(0, 2, 1)  # [B, W', 512] -> [B, T, C]
        
        # Add positional encoding
        features = self.pos_encoder(features)
        
        # Transformer encoding
        features = self.transformer_encoder(features)  # [B, T, 512]
        
        # CTC classification
        logits = self.classifier(features)  # [B, T, num_classes]
        
        return logits


# Initialize model
model = HTRModel(num_classes=num_classes + 1, config=config).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\nHTRModel initialized:")
print(f"  Architecture: CNN → Transformer → CTC")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / 1024**2:.2f} MB (float32)")
print(f"\nModel architecture:")
print(f"  CNN: 6 blocks, output channels=512")
print(f"  Transformer: {config.num_encoder_layers} layers, d_model={config.d_model}, heads={config.num_heads}")
print(f"  CTC output: {num_classes + 1} classes (including blank)")

In [ ]:
# Cell 10: Training utilities (ctc_decode_greedy, calculate_cer, calculate_wer, EarlyStopping)
def ctc_decode_greedy(logits, idx_to_char, blank_idx):
    """
    Greedy CTC decoding
    
    Args:
        logits: [T, num_classes] log probabilities
        idx_to_char: mapping from index to character
        blank_idx: CTC blank index
    
    Returns:
        decoded string
    """
    # Get best path
    best_path = torch.argmax(logits, dim=-1)  # [T]
    
    # Collapse repeats and remove blanks
    chars = []
    prev_idx = None
    for idx in best_path:
        idx = idx.item()
        if idx != blank_idx and idx != prev_idx:
            if idx in idx_to_char:
                chars.append(idx_to_char[idx])
        prev_idx = idx
    
    return ''.join(chars)


def calculate_cer(pred: str, target: str) -> float:
    """Calculate Character Error Rate"""
    if len(target) == 0:
        return 1.0 if len(pred) > 0 else 0.0
    return Levenshtein.distance(pred, target) / len(target)


def calculate_wer(pred: str, target: str) -> float:
    """Calculate Word Error Rate"""
    pred_words = pred.split()
    target_words = target.split()
    if len(target_words) == 0:
        return 1.0 if len(pred_words) > 0 else 0.0
    return Levenshtein.distance(pred_words, target_words) / len(target_words)


class EarlyStopping:
    """Early stopping to stop training when validation loss doesn't improve"""
    
    def __init__(self, patience: int = 10, min_delta: float = 0.0, mode: str = 'min'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_epoch = 0
    
    def __call__(self, score, epoch):
        if self.best_score is None:
            self.best_score = score
            self.best_epoch = epoch
            return False
        
        if self.mode == 'min':
            improved = score < (self.best_score - self.min_delta)
        else:
            improved = score > (self.best_score + self.min_delta)
        
        if improved:
            self.best_score = score
            self.best_epoch = epoch
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        
        return self.early_stop


print("Training utilities defined:")
print("  - ctc_decode_greedy: Greedy CTC decoding")
print("  - calculate_cer: Character Error Rate")
print("  - calculate_wer: Word Error Rate")
print("  - EarlyStopping: Early stopping callback")

In [ ]:
# Cell 11: Training setup (optimizer, scheduler, criterion)
# Optimizer: AdamW with weight decay
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

# Learning rate scheduler: CosineAnnealingLR with warmup
num_training_steps = len(train_loader) * config.NUM_EPOCHS
num_warmup_steps = config.warmup_steps

def lr_lambda(current_step):
    if current_step < num_warmup_steps:
        return float(current_step) / float(max(1, num_warmup_steps))
    progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
    return max(0.0, 0.5 * (1.0 + np.cos(np.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# CTC Loss criterion
ctc_criterion = nn.CTCLoss(blank=num_classes, zero_infinity=True)

# Early stopping
early_stopping = EarlyStopping(patience=config.patience, mode='min')

print("Training setup complete:")
print(f"  Optimizer: AdamW (lr={config.LEARNING_RATE}, weight_decay={config.WEIGHT_DECAY})")
print(f"  Scheduler: Cosine with warmup ({num_warmup_steps} steps)")
print(f"  Criterion: CTCLoss (blank={num_classes})")
print(f"  Early stopping: patience={config.patience}")
print(f"  Gradient clipping: max_norm={config.GRADIENT_CLIP}")
print(f"  Total training steps: {num_training_steps}")

In [ ]:
# Cell 12: train_epoch and validate_epoch functions
def train_epoch(model, train_loader, optimizer, criterion, scheduler, device, epoch):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    total_cer = 0
    total_wer = 0
    num_batches = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]", leave=False)
    
    for batch_idx, (images, labels, label_lengths, texts) in enumerate(pbar):
        images = images.to(device)
        labels = labels.to(device)
        label_lengths = label_lengths.to(device)
        
        # Forward pass
        logits = model(images)  # [B, T, num_classes]
        
        # Prepare for CTC loss: [T, B, num_classes]
        log_probs = F.log_softmax(logits, dim=-1)
        log_probs = log_probs.permute(1, 0, 2)  # [T, B, num_classes]
        
        # Input lengths (time steps)
        input_lengths = torch.full(
            size=(images.size(0),), 
            fill_value=log_probs.size(0), 
            dtype=torch.long,
            device=device
        )
        
        # CTC loss
        loss = criterion(log_probs, labels, input_lengths, label_lengths)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
        
        optimizer.step()
        scheduler.step()
        
        # Calculate metrics
        total_loss += loss.item()
        
        # Decode predictions for metrics
        with torch.no_grad():
            for i in range(logits.size(0)):
                pred_text = ctc_decode_greedy(logits[i], idx_to_char, num_classes)
                target_text = texts[i]
                total_cer += calculate_cer(pred_text, target_text)
                total_wer += calculate_wer(pred_text, target_text)
        
        num_batches += 1
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'lr': f"{scheduler.get_last_lr()[0]:.6f}"
        })
    
    avg_loss = total_loss / num_batches
    avg_cer = total_cer / (num_batches * config.BATCH_SIZE)
    avg_wer = total_wer / (num_batches * config.BATCH_SIZE)
    
    return avg_loss, avg_cer, avg_wer


def validate_epoch(model, val_loader, criterion, device, epoch):
    """Validate for one epoch"""
    model.eval()
    total_loss = 0
    total_cer = 0
    total_wer = 0
    num_batches = 0
    
    predictions = []
    
    pbar = tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]", leave=False)
    
    with torch.no_grad():
        for images, labels, label_lengths, texts in pbar:
            images = images.to(device)
            labels = labels.to(device)
            label_lengths = label_lengths.to(device)
            
            # Forward pass
            logits = model(images)  # [B, T, num_classes]
            
            # Prepare for CTC loss
            log_probs = F.log_softmax(logits, dim=-1)
            log_probs = log_probs.permute(1, 0, 2)  # [T, B, num_classes]
            
            input_lengths = torch.full(
                size=(images.size(0),),
                fill_value=log_probs.size(0),
                dtype=torch.long,
                device=device
            )
            
            # CTC loss
            loss = criterion(log_probs, labels, input_lengths, label_lengths)
            total_loss += loss.item()
            
            # Decode predictions
            for i in range(logits.size(0)):
                pred_text = ctc_decode_greedy(logits[i], idx_to_char, num_classes)
                target_text = texts[i]
                total_cer += calculate_cer(pred_text, target_text)
                total_wer += calculate_wer(pred_text, target_text)
                predictions.append((pred_text, target_text))
            
            num_batches += 1
            
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
    
    avg_loss = total_loss / num_batches
    avg_cer = total_cer / (num_batches * config.BATCH_SIZE)
    avg_wer = total_wer / (num_batches * config.BATCH_SIZE)
    
    return avg_loss, avg_cer, avg_wer, predictions


print("Training functions defined:")
print("  - train_epoch: Train for one epoch with CTC loss")
print("  - validate_epoch: Validate and compute metrics")

In [ ]:
# Cell 13: Main training loop (30 epochs)
print(f"Starting training for {config.NUM_EPOCHS} epochs...\n")

# Training history
history = {
    'train_loss': [],
    'train_cer': [],
    'train_wer': [],
    'val_loss': [],
    'val_cer': [],
    'val_wer': [],
    'learning_rates': []
}

best_val_cer = float('inf')
best_epoch = 0

for epoch in range(config.NUM_EPOCHS):
    print(f"\n{'='*80}")
    print(f"Epoch {epoch+1}/{config.NUM_EPOCHS}")
    print(f"{'='*80}")
    
    # Train
    train_loss, train_cer, train_wer = train_epoch(
        model, train_loader, optimizer, ctc_criterion, scheduler, device, epoch
    )
    
    # Validate
    val_loss, val_cer, val_wer, val_predictions = validate_epoch(
        model, val_loader, ctc_criterion, device, epoch
    )
    
    # Record history
    history['train_loss'].append(train_loss)
    history['train_cer'].append(train_cer)
    history['train_wer'].append(train_wer)
    history['val_loss'].append(val_loss)
    history['val_cer'].append(val_cer)
    history['val_wer'].append(val_wer)
    history['learning_rates'].append(scheduler.get_last_lr()[0])
    
    # Print metrics
    print(f"\nTrain - Loss: {train_loss:.4f}, CER: {train_cer:.4f}, WER: {train_wer:.4f}")
    print(f"Val   - Loss: {val_loss:.4f}, CER: {val_cer:.4f}, WER: {val_wer:.4f}")
    print(f"LR: {scheduler.get_last_lr()[0]:.6f}")
    
    # Save best model
    if val_cer < best_val_cer:
        best_val_cer = val_cer
        best_epoch = epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_cer': val_cer,
            'val_wer': val_wer,
            'config': config
        }, os.path.join(config.output_dir, 'best_model.pt'))
        print(f"Best model saved (CER: {val_cer:.4f})")
    
    # Show sample predictions
    if len(val_predictions) > 0:
        print(f"\nSample predictions:")
        for i in range(min(3, len(val_predictions))):
            pred, target = val_predictions[i]
            print(f"  Pred: {pred[:100]}")
            print(f"  True: {target[:100]}")
            print()
    
    # Early stopping check
    if early_stopping(val_cer, epoch):
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        print(f"Best epoch: {best_epoch+1} with CER: {best_val_cer:.4f}")
        break

print(f"\n{'='*80}")
print("Training completed!")
print(f"Best validation CER: {best_val_cer:.4f} at epoch {best_epoch+1}")
print(f"{'='*80}")

In [ ]:
# Cell 14: Create training_overview.png visualization
print("Creating training overview visualization...\n")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
epochs_range = range(1, len(history['train_loss']) + 1)

# Loss plot
axes[0, 0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
axes[0, 0].plot(epochs_range, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('CTC Loss', fontsize=12)
axes[0, 0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axvline(x=best_epoch+1, color='g', linestyle='--', alpha=0.5, label=f'Best (Epoch {best_epoch+1})')

# CER plot
axes[0, 1].plot(epochs_range, history['train_cer'], 'b-', label='Train CER', linewidth=2)
axes[0, 1].plot(epochs_range, history['val_cer'], 'r-', label='Val CER', linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Character Error Rate', fontsize=12)
axes[0, 1].set_title('Character Error Rate (CER)', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axvline(x=best_epoch+1, color='g', linestyle='--', alpha=0.5)
axes[0, 1].axhline(y=best_val_cer, color='g', linestyle='--', alpha=0.5, label=f'Best CER: {best_val_cer:.4f}')

# WER plot
axes[1, 0].plot(epochs_range, history['train_wer'], 'b-', label='Train WER', linewidth=2)
axes[1, 0].plot(epochs_range, history['val_wer'], 'r-', label='Val WER', linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Word Error Rate', fontsize=12)
axes[1, 0].set_title('Word Error Rate (WER)', fontsize=14, fontweight='bold')
axes[1, 0].legend(fontsize=11)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].axvline(x=best_epoch+1, color='g', linestyle='--', alpha=0.5)

# Learning rate plot
axes[1, 1].plot(epochs_range, history['learning_rates'], 'purple', linewidth=2)
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Learning Rate', fontsize=12)
axes[1, 1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_yscale('log')
axes[1, 1].axvline(x=best_epoch+1, color='g', linestyle='--', alpha=0.5)

plt.suptitle(
    f'Training Overview - Best Val CER: {best_val_cer:.4f} (Epoch {best_epoch+1})',
    fontsize=16,
    fontweight='bold',
    y=0.995
)
plt.tight_layout()

# Save figure
overview_path = os.path.join(config.charts_dir, 'training', 'training_overview.png')
plt.savefig(overview_path, dpi=300, bbox_inches='tight')
print(f"Training overview saved to: {overview_path}")
plt.show()

# Print final summary
print("\nTraining Summary:")
print(f"  Total epochs: {len(history['train_loss'])}")
print(f"  Best validation CER: {best_val_cer:.4f} at epoch {best_epoch+1}")
print(f"  Final train CER: {history['train_cer'][-1]:.4f}")
print(f"  Final val CER: {history['val_cer'][-1]:.4f}")
print(f"  Final train WER: {history['train_wer'][-1]:.4f}")
print(f"  Final val WER: {history['val_wer'][-1]:.4f}")

In [ ]:
# Cell 15: Test set evaluation
print("Evaluating on test set...\n")

# Load best model
checkpoint = torch.load(os.path.join(config.output_dir, 'best_model.pt'))
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")

# Evaluate on test set
test_loss, test_cer, test_wer, test_predictions = validate_epoch(
    model, test_loader, ctc_criterion, device, epoch=0
)

print(f"\n{'='*80}")
print("TEST SET RESULTS")
print(f"{'='*80}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test CER: {test_cer:.4f}")
print(f"Test WER: {test_wer:.4f}")
print(f"{'='*80}")

# Show sample predictions
print(f"\nSample test predictions:")
print(f"{'-'*80}")
for i in range(min(5, len(test_predictions))):
    pred, target = test_predictions[i]
    cer = calculate_cer(pred, target)
    print(f"\nSample {i+1}:")
    print(f"  Predicted: {pred[:150]}")
    print(f"  Target:    {target[:150]}")
    print(f"  CER: {cer:.4f}")
    print(f"{'-'*80}")

# Save test results
test_results = {
    'test_loss': test_loss,
    'test_cer': test_cer,
    'test_wer': test_wer,
    'predictions': test_predictions
}

test_results_path = os.path.join(config.output_dir, 'test_results.json')
with open(test_results_path, 'w', encoding='utf-8') as f:
    json.dump({
        'test_loss': test_loss,
        'test_cer': test_cer,
        'test_wer': test_wer,
        'num_samples': len(test_predictions)
    }, f, ensure_ascii=False, indent=2)

print(f"\nTest results saved to: {test_results_path}")

In [ ]:
# Cell 16: Sample predictions visualization
print("Creating sample predictions visualization...\n")

# Select diverse samples for visualization
num_samples = min(6, len(test_predictions))
sample_indices = np.linspace(0, len(test_predictions)-1, num_samples, dtype=int)

fig, axes = plt.subplots(num_samples, 1, figsize=(16, 3*num_samples))
if num_samples == 1:
    axes = [axes]

for idx, sample_idx in enumerate(sample_indices):
    pred, target = test_predictions[sample_idx]
    cer = calculate_cer(pred, target)
    wer = calculate_wer(pred, target)
    
    ax = axes[idx]
    ax.axis('off')
    
    # Format text for display
    max_len = 120
    pred_display = pred[:max_len] + ('...' if len(pred) > max_len else '')
    target_display = target[:max_len] + ('...' if len(target) > max_len else '')
    
    # Color based on CER
    if cer < 0.1:
        color = 'green'
        quality = 'Excellent'
    elif cer < 0.3:
        color = 'orange'
        quality = 'Good'
    else:
        color = 'red'
        quality = 'Poor'
    
    text_content = f"""Sample {idx+1} - CER: {cer:.4f} ({quality}) | WER: {wer:.4f}

Predicted: {pred_display}

Ground Truth: {target_display}
"""
    
    ax.text(0.05, 0.5, text_content, 
            fontsize=10, 
            verticalalignment='center',
            family='monospace',
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=color, linewidth=2, alpha=0.8))

plt.suptitle(
    f'Sample Predictions (Test Set) - Average CER: {test_cer:.4f}',
    fontsize=16,
    fontweight='bold',
    y=0.995
)
plt.tight_layout()

# Save figure
predictions_path = os.path.join(config.charts_dir, 'predictions', 'sample_predictions.png')
plt.savefig(predictions_path, dpi=200, bbox_inches='tight')
print(f"Sample predictions saved to: {predictions_path}")
plt.show()

In [ ]:
# Cell 17: Save all artifacts (model, metrics, config, history)
print("Saving final artifacts...\n")

# Save training history
history_path = os.path.join(config.output_dir, 'training_history.json')
with open(history_path, 'w', encoding='utf-8') as f:
    json.dump(history, f, indent=2)
print(f"Training history saved to: {history_path}")

# Save configuration
config_dict = {
    'IMG_HEIGHT': config.IMG_HEIGHT,
    'BATCH_SIZE': config.BATCH_SIZE,
    'NUM_EPOCHS': config.NUM_EPOCHS,
    'LEARNING_RATE': config.LEARNING_RATE,
    'WEIGHT_DECAY': config.WEIGHT_DECAY,
    'd_model': config.d_model,
    'num_encoder_layers': config.num_encoder_layers,
    'num_heads': config.num_heads,
    'dim_feedforward': config.dim_feedforward,
    'dropout': config.dropout,
    'num_classes': num_classes + 1,
    'vocab_size': len(vocab),
    'device': str(device),
    'total_params': total_params,
    'trainable_params': trainable_params
}

config_path = os.path.join(config.output_dir, 'config.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config_dict, f, indent=2)
print(f"Configuration saved to: {config_path}")

# Save final model
final_model_path = os.path.join(config.output_dir, 'final_model.pt')
torch.save({
    'epoch': len(history['train_loss']) - 1,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'test_cer': test_cer,
    'test_wer': test_wer,
    'config': config_dict
}, final_model_path)
print(f"Final model saved to: {final_model_path}")

# Create summary report
summary = f"""Azerbaijani HTR Training Summary
{'='*80}

CONFIGURATION:
  Architecture: CNN → Transformer → CTC
  IMG_HEIGHT: {config.IMG_HEIGHT}
  BATCH_SIZE: {config.BATCH_SIZE}
  Transformer: d_model={config.d_model}, layers={config.num_encoder_layers}, heads={config.num_heads}
  Total parameters: {total_params:,}
  Device: {device}

DATASET:
  Train samples: {len(train_dataset)}
  Val samples: {len(val_dataset)}
  Test samples: {len(test_dataset)}
  Vocabulary size: {len(vocab)}
  Classes (with CTC blank): {num_classes + 1}

TRAINING RESULTS:
  Total epochs: {len(history['train_loss'])}
  Best validation CER: {best_val_cer:.4f} (epoch {best_epoch+1})
  Final train CER: {history['train_cer'][-1]:.4f}
  Final val CER: {history['val_cer'][-1]:.4f}

TEST SET PERFORMANCE:
  Test Loss: {test_loss:.4f}
  Test CER: {test_cer:.4f}
  Test WER: {test_wer:.4f}

ARTIFACTS:
  Best model: {os.path.join(config.output_dir, 'best_model.pt')}
  Final model: {final_model_path}
  Vocabulary: {vocab_path}
  Configuration: {config_path}
  Training history: {history_path}
  Test results: {test_results_path}
  Training overview: {overview_path}
  Sample predictions: {predictions_path}

{'='*80}
"""

summary_path = os.path.join(config.output_dir, 'training_summary.txt')
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(summary)
print(f"Training summary saved to: {summary_path}")

print("\n" + summary)
print("\nAll artifacts saved successfully!")

## Training Complete!

### Next Steps:

1. **Model Deployment**: Export the best model to ONNX for production inference
2. **Language Model Integration**: Add KenLM-based rescoring for improved accuracy
3. **Error Analysis**: Analyze character confusion matrix and common errors
4. **Data Augmentation**: If CER is high, add more aggressive augmentation
5. **Active Learning**: Label more samples focusing on high-error cases

### Key Files:
- `outputs/best_model.pt` - Best model by validation CER
- `outputs/vocabulary.json` - Character-to-index mapping
- `outputs/config.json` - Model configuration
- `charts/training/training_overview.png` - Training metrics visualization
- `charts/predictions/sample_predictions.png` - Sample predictions
- `charts/preprocessing/` - Preprocessing stage visualizations